# FinGPT-Forecaster Re-implemented with finrobot_zh

In this demo, we set up an agent to emulate the behavior of model in the fingpt-forecaster project with AutoGen, which takes a company's ticker symbol, recent basic financials and market news as input and predict its stock movements.

For detail of the original project, check out  [FinGPT-Forecaster](https://github.com/AI4Finance-Foundation/FinGPT/tree/master/fingpt/FinGPT_Forecaster)!  🔥[Demo](https://huggingface.co/spaces/FinGPT/FinGPT-Forecaster), [Medium Blog](https://medium.datadriveninvestor.com/introducing-fingpt-forecaster-the-future-of-robo-advisory-services-50add34e3d3c) & [Model](https://huggingface.co/FinGPT/fingpt-forecaster_dow30_llama2-7b_lora) on Huggingface🤗!

In [1]:
!pip install -r /Users/yaolo/Desktop/FinRobot/requirements.txt #替換成實際路徑位置

In [2]:
import sys
sys.path.append("/Users/yaolo/Desktop/FinRobot") #替換成實際路徑位置

In [3]:
import autogen
from autogen.cache import Cache

from finrobot_zh.utils import get_current_date, register_keys_from_json
from finrobot_zh.data_source import FinnHubUtils, YFinanceUtils

After importing all the necessary packages and functions, we first instantiate a market analysis assistant and a user proxy using the agent classes defined by AutoGen. We also need the config for OpenAI & Finnhub here.
- for openai configuration, rename OAI_CONFIG_LIST_sample to OAI_CONFIG_LIST and replace the api keys
- for finnhub configuration, rename config_api_keys_sample to config_api_keys and replace the api keys

In [4]:
# Read OpenAI API keys from a JSON file
config_list = autogen.config_list_from_json(
    "/Users/yaolo/Desktop/FinRobot/OAI_CONFIG_LIST", #替換成實際路徑位置
    filter_dict={"model": ["gpt-4o-mini"]},
)
llm_config = {"config_list": config_list, "timeout": 120, "temperature": 0}

# Register FINNHUB API keys
register_keys_from_json("/Users/yaolo/Desktop/FinRobot/config_api_keys") #替換成實際路徑位置

In [5]:
analyst = autogen.AssistantAgent(
    name="Market_Analyst",
    system_message='''身為一名市場分析師，必須具備出色的分析與解決問題的能力，能夠根據客戶需求蒐集必要的財務資訊並進行整合。
    在處理程式設計任務時，僅限使用所提供的函數完成工作。任務結束後請回覆「TERMINATE」。''',
    llm_config=llm_config,
)

user_proxy = autogen.UserProxyAgent(
    name="User_Proxy",
    is_termination_msg=lambda x: x.get("content", "") and x.get(
        "content", "").endswith("TERMINATE"),
    human_input_mode="NEVER",
    max_consecutive_auto_reply=10,
    code_execution_config={
        "work_dir": "coding",
        "use_docker": False,
    },  # Please set use_docker=True if docker is available to run the generated code. Using docker is safer than running the generated code directly.
)

Then we register our predefined functions with the userproxy.

Following our implementation with the fingpt-forecaster, we combined news/financials calls to the Finnhub API and stock data calls to the YFinance API.

In [6]:
from finrobot_zh.toolkits import register_toolkits

tools = [
    {
        "function": FinnHubUtils.get_company_profile,
        "name": "get_company_profile",
        "description": "get a company's profile information"
    },
    {
        "function": FinnHubUtils.get_company_news,
        "name": "get_company_news",
        "description": "retrieve market news related to designated company"
    },
    {
        "function": FinnHubUtils.get_basic_financials,
        "name": "get_financial_basics",
        "description": "get latest financial basics for a designated company"
    },
    {
        "function": YFinanceUtils.get_stock_data,
        "name": "get_stock_data",
        "description": "retrieve stock price data for designated ticker symbol"
    }
]
register_toolkits(tools, analyst, user_proxy)

Start the conversation, asking the agent to call all the functions, and see how it aggregates all the information and leads to the conclusion.

In [7]:
# company = "Tesla"
company = "BlackRock"

with Cache.disk() as cache:
    # start the conversation
    user_proxy.initiate_chat(
        analyst,
        message=
        f"請使用所有提供的工具，根據 {get_current_date()}，蒐集關於 {company} 的相關資訊。"
        f"分析 {company} 的正面發展及潛在問題，並針對這兩方面分別列出 2 至 4 個最重要的因素。"
        "這些因素應主要從與公司相關的新聞中推斷出來，請保持內容簡潔。"
        f"接著，對 {company} 的下週股價走勢進行粗略預測（例如：上漲/下跌 2-3%）。"
        "最後，提供一份簡要分析，說明支持您預測的依據。",
        cache=cache,
    )

User_Proxy (to Market_Analyst):

請使用所有提供的工具，根據 2024-12-22，蒐集關於 BlackRock 的相關資訊。分析 BlackRock 的正面發展及潛在問題，並針對這兩方面分別列出 2 至 4 個最重要的因素。這些因素應主要從與公司相關的新聞中推斷出來，請保持內容簡潔。接著，對 BlackRock 的下週股價走勢進行粗略預測（例如：上漲/下跌 2-3%）。最後，提供一份簡要分析，說明支持您預測的依據。

--------------------------------------------------------------------------------
Market_Analyst (to User_Proxy):

***** Suggested tool call (call_Yx7WTR9THyMZoX4c1GD1lnAJ): get_company_profile *****
Arguments: 
{"symbol": "BLK"}
************************************************************************************
***** Suggested tool call (call_bLoTLWl5TraHvZQ8BhKzTEJc): get_company_news *****
Arguments: 
{"start_date": "2024-12-15", "end_date": "2024-12-22", "symbol": "BLK"}
*********************************************************************************
***** Suggested tool call (call_gaxsxRL7YG5YadAulhyPXQnW): get_stock_data *****
Arguments: 
{"start_date": "2024-12-15", "end_date": "2024-12-22", "symbol": "BLK"}
**********************************************